# Fashion-MNIST: clasificación CNN y robustez adversaria con FGSM

Este notebook reproduce el protocolo implementado en `src/fashion_mnist_fgsm.py`: entrenamiento limpio, generación de ejemplos FGSM y evaluación de accuracy robusta y ASR.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src').exists():
    raise RuntimeError('Ejecute el notebook desde la raíz del repositorio o desde notebooks/.')
sys.path.insert(0, str(ROOT))

from src.fashion_mnist_fgsm import (
    EPSILON_VALUES,
    build_model,
    configure_reproducibility,
    evaluate_epsilons,
    load_fashion_mnist,
    plot_metrics,
    train_model,
)

configure_reproducibility()

## 1. Carga y preprocesamiento

TensorFlow descarga Fashion-MNIST automáticamente. Los píxeles se normalizan a `[0, 1]` y se agrega el canal de escala de grises.

In [ ]:
x_train, y_train, x_test, y_test = load_fashion_mnist()
print('Entrenamiento:', x_train.shape, y_train.shape)
print('Prueba:', x_test.shape, y_test.shape)

## 2. Construcción y entrenamiento de la CNN

La salida contiene diez logits y la pérdida utiliza `from_logits=True`.

In [ ]:
model = build_model()
model.summary()

history = train_model(
    model,
    x_train,
    y_train,
    epochs=20,
    batch_size=128,
)

In [ ]:
test_loss, clean_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f'Accuracy limpia: {clean_accuracy:.4f}')

## 3. Evaluación adversaria FGSM

Se utiliza el mismo modelo para todos los valores de epsilon. Los pesos permanecen fijos.

In [ ]:
results = evaluate_epsilons(
    model,
    x_test,
    y_test,
    EPSILON_VALUES,
)

print('epsilon | accuracy robusta | ASR')
for row in results:
    print(
        f"{row['epsilon']:7.2f} | "
        f"{row['robust_accuracy']:16.4f} | "
        f"{row['attack_success_rate']:7.4f}"
    )

In [ ]:
from IPython.display import Image, display

figure_path = ROOT / 'results' / 'generated' / 'notebook_fgsm_metrics.png'
plot_metrics(results, figure_path)
display(Image(filename=str(figure_path)))

## Interpretación

La accuracy robusta debe coincidir con la accuracy limpia cuando `epsilon = 0`. Al aumentar epsilon se espera una reducción de la robustez y un incremento de la ASR. FGSM diagnostica vulnerabilidad; no constituye una defensa.